<a href="https://colab.research.google.com/github/thinus283-ux/LR/blob/main/175_SPARC_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ================================================
# LR v5.4 - Log-Scaled sigma_c + Multi-Start Rescue
# ================================================

import numpy as np
import zipfile
import os
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings("ignore")

def stress_profile(r, P0, beta, r_max, rim_power):
    base = P0 * np.exp(-beta * r)
    rim = 1 + 9.5 * (r / r_max)**rim_power
    return base * rim

def vortex_clamping(r, sigma, log_sigma_c, r_max, K, n, trans_fraction):
    sigma_c = 10 ** log_sigma_c
    excess = np.maximum(sigma - sigma_c, 0)
    r_trans = r_max * trans_fraction
    r_scale = np.maximum(r_max / 4.5, 1.2)
    rim_boost = 1 + 5.2 / (1 + np.exp(-(r - r_trans) / r_scale))
    return K * (excess ** n) * rim_boost

def fit_sparc_galaxy(r, Vobs, Verr, Vgas, Vdisk, Vbul):
    r_max = np.max(r)

    has_bulge = np.max(Vbul) > 0 if len(Vbul) > 0 else False
    max_disk = np.max(Vdisk) if len(Vdisk) > 0 else 0
    max_gas = np.max(Vgas) if len(Vgas) > 0 else 0

    def model(r, P0, beta, log_sigma_c, Yd, Yb, K, n, trans_fraction, rim_power):
        Vbar = np.sqrt(np.maximum(Vgas**2 + Yd * Vdisk**2 + Yb * Vbul**2, 0))
        sigma = stress_profile(r, P0, beta, r_max, rim_power)
        v_vortex = vortex_clamping(r, sigma, log_sigma_c, r_max, K, n, trans_fraction)
        return np.sqrt(Vbar**2 + v_vortex**2)

    # Morphology-adaptive baseline
    if has_bulge and np.max(Vbul) > 0.5 * max_disk:
        p0_base = [45000, 0.75, -11.0, 0.7, 0.8, 5.2, 0.42, 0.40, 3.2]
        trans_bounds = [0.20, 0.65]
        rim_bounds = [2.2, 4.5]
    elif max_disk > 2.0 * max_gas:
        p0_base = [28000, 0.45, -11.4, 0.55, 0.0, 4.8, 0.38, 0.55, 2.9]
        trans_bounds = [0.35, 0.75]
        rim_bounds = [2.4, 4.0]
    else:
        p0_base = [12000, 0.25, -12.1, 0.2, 0.0, 3.5, 0.32, 0.70, 2.6]
        trans_bounds = [0.50, 0.90]
        rim_bounds = [2.0, 3.8]

    lower = [100, 0.005, -16.0, 0.0, 0.0, 0.5, 0.05, trans_bounds[0], rim_bounds[0]]
    upper = [350000, 6.0, -4.0, 3.0, 3.0, 20.0, 0.95, trans_bounds[1], rim_bounds[1]]
    bounds = (lower, upper)

    def run_fit(initial_guess):
        popt, _ = curve_fit(
            model, r, Vobs,
            p0=initial_guess,
            sigma=Verr,
            bounds=bounds,
            absolute_sigma=True,
            loss='soft_l1',
            maxfev=60000
        )
        fit_rms = np.sqrt(np.mean((model(r, *popt) - Vobs)**2))
        return fit_rms, popt

    try:
        best_rms, best_popt = run_fit(p0_base)

        # Multi-start rescue if poor fit
        if best_rms > 12.0:
            alt_seeds = [
                [80000, 0.9, -9.0, 1.0, 1.0, 7.0, 0.45, (trans_bounds[0]+trans_bounds[1])/2, (rim_bounds[0]+rim_bounds[1])/2],
                [5000, 0.1, -14.0, 0.1, 0.0, 2.0, 0.25, trans_bounds[1]-0.05, rim_bounds[0]+0.2]
            ]
            for alt_p0 in alt_seeds:
                try:
                    alt_rms, alt_popt = run_fit(alt_p0)
                    if alt_rms < best_rms:
                        best_rms = alt_rms
                        best_popt = alt_popt
                except:
                    continue

        return best_rms, best_popt
    except:
        return np.nan, None

# ====================== FULL RUN WITH STATS ======================
print("Downloading official Rotmod_LTG.zip ...")
!wget -q https://astroweb.case.edu/SPARC/Rotmod_LTG.zip -O Rotmod_LTG.zip

with zipfile.ZipFile("Rotmod_LTG.zip", 'r') as zip_ref:
    zip_ref.extractall("sparc_data")

dat_files = [f for f in os.listdir("sparc_data") if f.endswith('.dat')]
print(f"Processing {len(dat_files)} real SPARC galaxies with LR v5.4...\n")

rms_list = []
under_3 = under_4 = under_5 = 0

for fname in dat_files:
    path = os.path.join("sparc_data", fname)
    try:
        data = np.loadtxt(path, skiprows=1)
        if data.shape[1] < 6: continue

        r     = data[:, 0]
        Vobs  = data[:, 1]
        Verr  = data[:, 2]
        Vgas  = data[:, 3]
        Vdisk = data[:, 4]
        Vbul  = data[:, 5]

        rms, popt = fit_sparc_galaxy(r, Vobs, Verr, Vgas, Vdisk, Vbul)
        if not np.isnan(rms):
            rms_list.append(rms)
            if rms < 3: under_3 += 1
            if rms < 4: under_4 += 1
            if rms < 5: under_5 += 1
    except:
        continue

rms_list = np.array(rms_list)

print("\n=== LR v5.4 Results ===")
print(f"Galaxies fitted: {len(rms_list)}")
print(f"Median RMS: {np.nanmedian(rms_list):.2f} km/s")
print(f"Mean RMS:   {np.nanmean(rms_list):.2f} km/s")
print(f"Best: {np.nanmin(rms_list):.2f} km/s | Worst: {np.nanmax(rms_list):.2f} km/s")

print("\n=== Galaxies under X km/s ===")
print(f"Under 3 km/s : {under_3}")
print(f"Under 4 km/s : {under_4}")
print(f"Under 5 km/s : {under_5}")

Processing 175 real SPARC galaxies with LR v5.4...


=== LR v5.4 Results ===
Galaxies fitted: 174
Median RMS: 4.82 km/s
Mean RMS:   6.73 km/s
Best: 0.09 km/s | Worst: 38.44 km/s

=== Galaxies under X km/s ===
Under 3 km/s : 43
Under 4 km/s : 66
Under 5 km/s : 89
